# Day 2, Day 4: Translating Python to C++ with LLMs

## Introduction
This notebook builds a small Gradio interface that sends a Python program to a selected language model, asks for equivalent C++ source, and saves the result as `main.cpp`. It also includes helpers to capture Python output and compile/run generated C++ locally; it does not compare the two outputs automatically.

Before starting, configure the API keys for the hosted providers you plan to use, install the notebook dependencies, and start Ollama if you select a local Ollama model. Review model-generated C++ before compiling or running it; the final cells execute programs on this machine.

## Workflow summary
1. Load credentials and create clients for the supported providers.
2. Collect system/compiler details and define the conversion prompts.
3. Use the Gradio app to generate and save C++ source.
4. Optionally capture the Python output and run the saved C++ program to compare their results manually.


In [1]:
# Import configuration, provider, UI, and process utilities used below.
import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess

In [2]:
# Load configured provider credentials and report availability without printing their values.
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print("OpenAI API key found and ready.")
else:
    print("OpenAI API Key not found.")

if anthropic_api_key:
    print("Anthropic API key found and ready.")
else: 
    print("Anthropic API Key not found.")

if google_api_key:
    print("Google API key found and ready.")
else: 
    print("Google API Key not found.")

if grok_api_key:
    print("Grok API key found and ready.")
else:
    print("Grok API Key not found.")

if groq_api_key:
    print("Groq API key found and ready.")
else:
    print("Groq API Key not found.")

if openrouter_api_key:
    print("OpenRouter API key found and ready.")
else:
    print("OpenRouter API Key not found.")


OpenAI API key found and ready.
Anthropic API key found and ready.
Google API key found and ready.
Grok API key found and ready.
Groq API key found and ready.
OpenRouter API key found and ready.


In [3]:
# Configure OpenAI-compatible clients for hosted providers and local Ollama.

openai = OpenAI()

anthropic_url = 'https://api.anthropic.com/v1/'
gemini_url = 'https://generativelanguage.googleapis.com/v1beta/openai/'
grok_url = 'https://api.x.ai/v1'
groq_url = 'https://api.groq.com/openai/v1'
ollama_url = 'http://localhost:11434/v1'
openrouter_url = 'https://openrouter.ai/api/v1'

anthropic = OpenAI(api_key=anthropic_api_key,
                   base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key,
                base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key,
              base_url=grok_url)
groq = OpenAI(api_key=groq_api_key,
              base_url=groq_url)
ollama = OpenAI(api_key='ollama',
                base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key,
                    base_url=openrouter_url)

In [4]:
# Pair each selectable model ID with the provider client that serves it.
models = ['gpt-6.1-sol',
          'claude-sonnet-5-5',
          'grok-4.3',
          'gemini-4-argon',
          'qwen2.5-coder',
          'deepseek-coder-v2',
          'gpt-oss:20b',
          'qwen/qwen3-coder-30b-a3b-instruct',
          'openai/gpt-oss-120b']
clients = {'gpt-6.1-sol': openai,
           'claude-sonnet-5-5': anthropic,
           'grok-4.3': grok,
           'gemini-4-argon': gemini,
           'qwen2.5-coder': ollama,
           'deepseek-coder-v2': ollama,
           'gpt-oss:20b': ollama,
           'openai/gpt-oss-120b': openrouter,
           'qwen/qwen3-coder-30b-a3b-instruct': openrouter}

In [5]:
# Capture machine details that help the model tailor its generated C++ and build choices.
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Darwin',
  'arch': 'arm64',
  'release': '27.0.0',
  'version': 'Darwin Kernel Version 27.0.0: Tue Aug 11 21:05:48 PDT 2026; root:xnu-13432.1.9~1/RELEASE_ARM64_T6041',
  'kernel': '27.0.0',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'arm64-apple-darwin27.0.0'},
 'package_managers': ['xcode-select (CLT)', 'brew'],
 'cpu': {'brand': 'Apple M4 Pro',
  'cores_logical': 14,
  'cores_physical': 14,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'Apple clang version 21.0.0 (clang-2100.3.34.2)',
   'g++': 'Apple clang version 21.0.0 (clang-2100.3.34.2)',
   'clang': 'Apple clang version 21.0.0 (clang-2100.3.34.2)',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': 'GNU Make 3.81'},
  'linkers': {'ld_lld': ''}}}

In [6]:
# Define the compiler invocation and executable command used by the local run step.
compile_command = ['clang++',
                   '-std=c++17',
                   '-Ofast',
                   '-mcpu=native',
                   '-flto=thin',
                   '-fvisibility=hidden',
                   '-DNDEBUG',
                   'main.cpp',
                   '-o',
                   'main']
run_command =['./main']

In [7]:
# Ask for output-compatible C++ and include machine/build context in each conversion request.
system_prompt = """ 
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f""" 
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [8]:
# Package the system and user instructions in the chat format expected by the API clients.
def messages_for(python):
    return [
        {
            'role': 'system',
            'content': system_prompt
        },
        {
            'role': 'user',
            'content': user_prompt_for(python)
        }
    ]


In [9]:
# Save the model's generated C++ to the source file named in compile_command.
def write_output(cpp):
    with open('main.cpp', 'w') as f:
        f.write(cpp)

In [10]:
# Request a translation from the selected provider, save it, and return it for display.
def port(model, python):
    client = clients[model]
    openai_reasoning_models = {'gpt-6.1-sol'}
    reasoning_effort = 'high' if model in openai_reasoning_models else None
    response = client.chat.completions.create(
        model=model,
        messages=messages_for(python),
        reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp', '').replace('```', '')
    write_output(reply)
    return reply

In [11]:
# Store a small reciprocal-series benchmark for comparing Python and generated C++ runs.
pi = """ 
import time

def calculate(iterations, param1, param2):
   result = 1.0
   for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result +=(1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [12]:
# Execute the supplied Python example while capturing its printed output.
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()

    except Exception as e:
        output = str(e)

    finally:
        sys.stdout = old_stdout

    return output

In [13]:
# Compile the saved C++ program and run it three times to show its reported timings.
def compile_and_run():
    try:
        subprocess.run(compile_command,
                       check=True,
                       text=True,
                       capture_output=True)
        print(subprocess.run(run_command,
                             check=True,
                             text=True,
                             capture_output=True).stdout)
        print(subprocess.run(run_command,
                             check=True,
                             text=True,
                             capture_output=True).stdout)
        print(subprocess.run(run_command,
                             check=True,
                             text=True,
                             capture_output=True).stdout)
    except subprocess.CalledProcessError as e:
        print(f"An error occurred: {e.stderr}")

In [14]:
# Present model selection, Python input, generated C++ output, and conversion action in Gradio.
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label='Python code:',
                            lines=28,
                            value=pi)
        cpp = gr.Textbox(label='C++ code:',
                         lines=28)
    with gr.Row():
        model = gr.Dropdown(models,
                            label='Select model',
                            value=models[0])
        convert = gr.Button("Convert code")
    convert.click(port,
                  inputs=[model, python],
                  outputs=[cpp])
ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [15]:
# Run the compile-and-execute step after the Gradio launch cell has returned.
compile_and_run()

Result: 3.141592656089
Execution Time: 0.227360 seconds

Result: 3.141592656089
Execution Time: 0.212724 seconds

Result: 3.141592656089
Execution Time: 0.211603 seconds



## Conclusion
The notebook now connects every selectable model to its intended client and documents the conversion, save, and local execution steps. The Python helper can capture the reference program's printed output, while the C++ helper compiles `main.cpp` and performs three runs; the results are not compared automatically, so check them manually for each generated translation. Treat the timing as a small local experiment, not a general performance guarantee.
